# CNN Sentiment Analysis with Streamlit

A small end-to-end NLP project that classifies a review as positive or negative using a Keras 1D CNN.

## Before running
- Place the original dataset at `data/cnn_text_classification.csv` or update the path in the data-loading cell.
- Expected columns: `text` and `label` (with `0 = Negative`, `1 = Positive`). An optional `id` column is ignored.
- Run the notebook from top to bottom to train and save `sentiment_model.keras` and generate `app.py`.
- The model's recorded test-split score is not a guarantee of real-world performance. The notebook's separate 32-sentence challenge set scored 75%; see the README for limitations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reproducible results
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

In [ ]:
# Change the filename if your CSV has a different name.
from pathlib import Path

DATA_PATH = Path("data/cnn_text_classification.csv")
if not DATA_PATH.exists():
    # Backward-compatible fallback when the CSV is beside the notebook.
    DATA_PATH = Path("cnn_text_classification.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Place cnn_text_classification.csv in the data/ folder "
        "or beside this notebook. See README.md for the expected schema and data-source note."
    )
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print(df.columns.tolist())

In [ ]:
# Keep only the text and label columns.
# Rename your columns here if your CSV uses different names.

df = df[["text", "label"]].copy()

# Remove missing or empty reviews.
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["label"] = pd.to_numeric(df["label"], errors="coerce")

df = df[
    (df["text"] != "") &
    (df["label"].isin([0, 1]))
].copy()

df["label"] = df["label"].astype("int32")

# Remove duplicate reviews.
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Clean dataset:", df.shape)
print("\nLabel distribution:")
display(df["label"].value_counts())

# Confirm your mapping before continuing.
# This notebook assumes 0 = Negative and 1 = Positive.

In [ ]:
# Split BEFORE adding our custom examples.
# This prevents our added examples from appearing in the test set.

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

In [ ]:
# Held-out test sentences. They are defined first and never used for training.
# 1 = Positive, 0 = Negative.
hard_tests = [
    ('The film was thrilling and beautifully shot.', 1),
    ('The story was dull and the ending was lazy.', 0),
    ('I was completely hooked from start to finish.', 1),
    ('The movie made me feel nothing.', 0),
    ('I did not enjoy this film.', 0),
    ('The acting was not convincing.', 0),
    ("The plot wasn't confusing.", 1),
    ('It was not a good movie.', 0),
    ("I couldn't stop laughing.", 1),
    ('The movie was not boring at all.', 1),
    ("I don't think this film is good.", 0),
    ('There was nothing wrong with this movie.', 1),
    ('The story was slow, but the performances were wonderful.', 1),
    ('The visuals were gorgeous, but the story was boring.', 0),
    ('Although the plot was predictable, the acting was brilliant.', 1),
    ('The beginning was great, however the ending was a disappointment.', 0),
    ('Despite a weak script, the film was entertaining.', 1),
    ('I absolutely hated every minute.', 0),
    ('The film was unbelievably entertaining.', 1),
    ('I felt bored the whole time.', 0),
    ('I laughed and cried, it was beautiful.', 1),
    ('What a waste of money.', 0),
    ('The film was amazng.', 1),
    ('The mvie was terible.', 0),
    ('The pizza was delicious.', 1),
    ('The service was awful.', 0),
    ('Oh great, another boring sequel.', 0),
    ('Fantastic, I fell asleep in ten minutes.', 0),
    ('I did not like this movie at all.', 0),
    ("It wasn't boring.", 1),
    ('The ending was weak, but I still loved the characters.', 1),
    ('Great cast, but I was bored the whole time.', 0),
]

In [ ]:
# Label mapping:
# 0 = Negative
# 1 = Positive

extra_examples = [

    # 1. Negation
    ("I do not dislike this movie.", 1),
    ("It was not bad at all.", 1),
    ("I wouldn't call it terrible.", 1),
    ("The film was not without its charm.", 1),
    ("I can't say I hated it.", 1),
    ("It wasn't the worst film I've seen.", 1),
    ("I did not expect to enjoy it this much.", 1),
    ("I never regretted watching it.", 1),
    ("There is nothing I would change about it.", 1),
    ("The movie was not remotely entertaining.", 0),
    ("I wouldn't recommend it to anyone.", 0),
    ("The movie is not worth your time.", 0),
    ("I have never been so disappointed.", 0),
    ("There was nothing enjoyable about it.", 0),
    ("The ending did not make any sense.", 0),

    # 2. Double negatives and subtle wording
    ("I cannot say this was a bad movie.", 1),
    ("It is not impossible to enjoy this film.", 1),
    ("I wouldn't say it was awful.", 1),
    ("It wasn't exactly impressive.", 0),
    ("The experience was less than satisfying.", 0),
    ("It could hardly be called entertaining.", 0),
    ("I don't think it deserves all the criticism.", 1),
    ("I can't say this film is particularly good.", 0),
    ("It is not the least bit interesting.", 0),
    ("I wouldn't go so far as to call it brilliant.", 0),

    # 3. Contrast: but, although, however
    ("The acting was great, but the plot was weak.", 0),
    ("The story was slow, yet the ending was satisfying.", 1),
    ("I disliked the beginning but loved the finale.", 1),
    ("The visuals were beautiful, but the characters felt empty.", 0),
    ("The cast was excellent and the script was smart.", 1),
    ("The soundtrack was lovely, although the film dragged.", 0),
    ("Despite its flaws, I really enjoyed it.", 1),
    ("It had a strong start but a terrible ending.", 0),
    ("The plot was simple, but the performances were moving.", 1),
    ("I liked the concept, but the execution was poor.", 0),
    ("The jokes were funny and the pacing was excellent.", 1),
    ("The costumes looked good, but everything else felt cheap.", 0),
    ("The film has problems, yet it won me over.", 1),
    ("I enjoyed the acting, although the story disappointed me.", 0),
    ("The first hour was dull, but the final act was fantastic.", 1),
    ("The premise was promising, but the movie went nowhere.", 0),
    ("The characters were believable and the dialogue felt natural.", 1),
    ("The plot was confusing, and the ending made it worse.", 0),
    ("The acting was average, but the story was excellent.", 1),
    ("The story was interesting, but the ending ruined everything.", 0),

    # 4. Intensifiers
    ("The acting was unbelievably good.", 1),
    ("This is one of my favourite films.", 1),
    ("The story was incredibly moving.", 1),
    ("The performances were exceptionally strong.", 1),
    ("I absolutely loved every minute.", 1),
    ("The movie was painfully boring.", 0),
    ("The dialogue was ridiculously bad.", 0),
    ("The ending was extremely disappointing.", 0),
    ("The film was unbearably slow.", 0),
    ("The characters were completely unconvincing.", 0),
    ("The cinematography was remarkably beautiful.", 1),
    ("The writing was shockingly weak.", 0),
    ("The cast delivered a genuinely wonderful performance.", 1),
    ("It was a thoroughly unpleasant experience.", 0),
    ("The story was surprisingly engaging.", 1),
    ("The film was deeply frustrating.", 0),
    ("The acting was exceptionally poor.", 0),
    ("The ending was absolutely perfect.", 1),
    ("The film was unbelievably dull.", 0),
    ("Every scene was genuinely entertaining.", 1),

    # 5. Sarcasm and irony
    ("Wonderful, another two hours wasted.", 0),
    ("Brilliant! I love being bored to tears.", 0),
    ("What a masterpiece, if you enjoy watching paint dry.", 0),
    ("The plot holes were the real stars of the movie.", 0),
    ("I laughed so hard that I nearly checked the time.", 0),
    ("A thrilling film, if falling asleep is thrilling.", 0),
    ("That ending was so good I wish I could forget it.", 0),
    ("Five stars for making me regret movie night.", 0),
    ("Oh great, another predictable story.", 0),
    ("I loved waiting forever for something to happen.", 0),
    ("A truly unforgettable disaster.", 0),
    ("Well, that was surprisingly delightful.", 1),
    ("I expected a disaster, but it was genuinely fun.", 1),
    ("This movie was a pleasant surprise.", 1),
    ("The film was so moving that I cared about the characters.", 1),

    # 6. Comparisons
    ("The sequel is better than the original.", 1),
    ("The remake is nowhere near as good as the original.", 0),
    ("This is the strongest film in the series.", 1),
    ("The earlier movie was far more entertaining.", 0),
    ("It is just as funny as the trailer promised.", 1),
    ("The film is weaker than its reviews suggest.", 0),
    ("This version improves on the book.", 1),
    ("The story is less interesting than the characters.", 0),
    ("This is the worst sequel I have watched.", 0),
    ("It surpassed all my expectations.", 1),

    # 7. Understatement and subtle opinions
    ("It left a little to be desired.", 0),
    ("The film is decent enough for one viewing.", 1),
    ("It is hardly a memorable experience.", 0),
    ("I expected more from such a talented cast.", 0),
    ("It was better than I had expected.", 1),
    ("The movie never quite comes together.", 0),
    ("It has rough edges but plenty of heart.", 1),
    ("The story is passable at best.", 0),
    ("It is not flawless, but genuinely entertaining.", 1),
    ("It was a pleasant surprise.", 1),
    ("The film has little to offer beyond its visuals.", 0),
    ("I have mixed feelings, but I enjoyed it overall.", 1),
    ("It wasn't a complete waste of time.", 1),
    ("The movie failed to live up to expectations.", 0),
    ("There are worse ways to spend an evening.", 1),

    # 8. Emotional reactions
    ("I cried because the story was so beautiful.", 1),
    ("The ending left me completely disappointed.", 0),
    ("I couldn't stop smiling throughout the film.", 1),
    ("The characters made me feel absolutely nothing.", 0),
    ("I was emotionally invested from the beginning.", 1),
    ("The movie made me regret watching it.", 0),
    ("It was heartwarming and deeply meaningful.", 1),
    ("The film was exhausting and depressing.", 0),
    ("I felt connected to every character.", 1),
    ("I felt relieved when the movie finally ended.", 0),

    # 9. Story and pacing
    ("The plot was clever and well developed.", 1),
    ("The story went nowhere for two hours.", 0),
    ("The pacing kept the film exciting.", 1),
    ("The narrative was confusing and poorly written.", 0),
    ("The ending tied everything together beautifully.", 1),
    ("The plot twists were obvious from the start.", 0),
    ("Every scene contributed to the story.", 1),
    ("The movie was repetitive and unnecessarily long.", 0),
    ("The story was original and refreshing.", 1),
    ("The screenplay was lazy and predictable.", 0),

    # 10. Acting and characters
    ("The lead actor delivered a powerful performance.", 1),
    ("The acting felt wooden and lifeless.", 0),
    ("The characters were believable and memorable.", 1),
    ("The cast lacked chemistry.", 0),
    ("The supporting actors stole every scene.", 1),
    ("The characters were shallow and annoying.", 0),
    ("The performances felt natural and sincere.", 1),
    ("The dialogue sounded artificial.", 0),
    ("The lead actress was absolutely captivating.", 1),
    ("Nobody in the cast seemed interested.", 0),

    # 11. Visuals and sound
    ("The cinematography was stunning.", 1),
    ("The special effects looked cheap.", 0),
    ("The soundtrack perfectly suited the story.", 1),
    ("The sound mixing was terrible.", 0),
    ("The sets were detailed and beautiful.", 1),
    ("The visuals were dull and uninspired.", 0),
    ("The editing was smooth and effective.", 1),
    ("The camera work was distracting.", 0),
    ("The colours and lighting were gorgeous.", 1),
    ("The production looked rushed and unfinished.", 0),

    # 12. Overall opinions
    ("I would happily watch it again.", 1),
    ("I wish I had chosen another movie.", 0),
    ("A charming film with plenty of heart.", 1),
    ("Nothing about this film held my attention.", 0),
    ("It delivered exactly the fun I wanted.", 1),
    ("The movie felt long, dull, and pointless.", 0),
    ("The emotional moments felt sincere.", 1),
    ("The jokes were awkward and rarely landed.", 0),
    ("A clever script with memorable performances.", 1),
    ("I struggled to finish it.", 0),

    # 13. "Waste of ..." / "worth ..." phrases
    ("The film was a waste of time.", 0),
    ("This was a complete waste of money.", 0),
    ("What a waste of time.", 0),
    ("The story was a total waste of talent.", 0),
    ("Such a waste of a good cast.", 0),
    ("A waste of time and money.", 0),
    ("The ending was a waste of everyone's time.", 0),
    ("It felt like a waste of an evening.", 0),
    ("I want my two hours back.", 0),
    ("Complete waste of my time.", 0),
    ("It was not worth the ticket price.", 0),
    ("The sequel was a pointless waste of time.", 0),
    ("It was well worth my time.", 1),
    ("The film was worth the ticket price.", 1),
    ("Definitely worth watching.", 1),
    ("The story was worth every minute.", 1),
    ("Not a waste of time at all.", 1),
    ("It was a great use of my evening.", 1),
    ("Worth every penny.", 1),
    ("The sequel was absolutely worth the wait.", 1),
    ("A film worth seeing twice.", 1),
    ("It was time well spent.", 1),
    ("I would gladly spend another two hours on it.", 1),
    ("The movie was worth it.", 1)
]

extra_df = pd.DataFrame(
    extra_examples,
    columns=["text", "label"]
)

print("Additional examples:", len(extra_df))
display(extra_df.head(10))

In [ ]:
# Systematic data augmentation (instead of adding one sentence at a time).
# We take clearly positive / negative words and build many sentences with the
# patterns the model keeps getting wrong: negation, contrast ("but"), and
# opinion verbs ("I did not like ..."). Labels follow simple, fixed rules.
import re
from collections import Counter

# 1) Words that appear (almost) only in one class of the TRAINING reviews.
pos_counts, neg_counts = Counter(), Counter()
for text, label in zip(train_df["text"], train_df["label"]):
    words = set(re.findall(r"[a-z']+", text.lower()))
    (pos_counts if label == 1 else neg_counts).update(words)

def one_sided(mine, other):
    return {w for w, c in mine.items() if c >= 5 and other[w] <= 0.05 * c}

# Add a few very common opinion words so the patterns also cover everyday wording.
pos_words = one_sided(pos_counts, neg_counts) | {"good", "great", "funny", "exciting", "entertaining", "interesting", "clever"}
neg_words = one_sided(neg_counts, pos_counts) | {"bad", "terrible", "slow", "annoying", "pointless", "boring", "dull"}
overlap_words = pos_words & neg_words
pos_words = sorted(pos_words - overlap_words)
neg_words = sorted(neg_words - overlap_words)
print("Positive words:", len(pos_words), "| Negative words:", len(neg_words))

rng = random.Random(SEED)
nouns = ["movie", "film", "story", "ending", "acting", "plot", "script", "dialogue"]
degrees = ["", "too ", "very ", "that ", "so ", "really "]
rows = []

# A) Negated opinions:  not + positive word = negative,  not + negative word = positive
for i in range(160):
    noun, degree = rng.choice(nouns), rng.choice(degrees)
    be = rng.choice(["was not", "wasn't", "is not", "isn't"])
    tail = rng.choice(["", "", " at all"])
    if i % 2 == 0:
        rows.append((f"The {noun} {be} {degree}{rng.choice(pos_words)}{tail}.", 0))
    else:
        rows.append((f"The {noun} {be} {degree}{rng.choice(neg_words)}{tail}.", 1))

# B) Contrast: the part after "but / yet / although" decides the label.
for i in range(160):
    n1, n2 = rng.sample(nouns, 2)
    if i % 2 == 0:
        a1, a2, label = rng.choice(pos_words), rng.choice(neg_words), 0
    else:
        a1, a2, label = rng.choice(neg_words), rng.choice(pos_words), 1
    style = rng.choice(["but", "yet", "although", "however"])
    if style == "although":
        rows.append((f"Although the {n1} was {a1}, the {n2} was {a2}.", label))
    elif style == "however":
        rows.append((f"The {n1} was {a1}, however the {n2} was {a2}.", label))
    else:
        rows.append((f"The {n1} was {a1}, {style} the {n2} was {a2}.", label))

# C) Opinion verbs: "I did not like it" is negative, "I did not hate it" is positive.
pos_verbs, neg_verbs = ["like", "love", "enjoy"], ["hate", "dislike"]
pos_past, neg_past = ["liked", "loved", "enjoyed"], ["hated", "disliked"]
objects = ["this movie", "this film", "the movie", "the story", "it", "the ending", "the acting"]
for i in range(160):
    obj, tail = rng.choice(objects), rng.choice(["", "", " at all", " very much"])
    did = rng.choice(["did not", "didn't"])
    kind = i % 4
    if kind == 0:
        rows.append((f"I {did} {rng.choice(pos_verbs)} {obj}{tail}.", 0))
    elif kind == 1:
        rows.append((f"I {did} {rng.choice(neg_verbs)} {obj}{tail}.", 1))
    elif kind == 2:
        rows.append((f"I {rng.choice(pos_past)} {obj}{tail}.", 1))
    else:
        rows.append((f"I {rng.choice(neg_past)} {obj}{tail}.", 0))

# D) Contrast with an opinion verb: "The ending was weak, but I loved the acting."
for i in range(80):
    n1, n2 = rng.sample(nouns, 2)
    if i % 2 == 0:
        rows.append((f"The {n1} was {rng.choice(neg_words)}, but I {rng.choice(['still ', ''])}{rng.choice(pos_past)} the {n2}.", 1))
    else:
        rows.append((f"The {n1} was {rng.choice(pos_words)}, but I {rng.choice(['still ', ''])}{rng.choice(neg_past)} the {n2}.", 0))

gen_df = pd.DataFrame(rows, columns=["text", "label"]).drop_duplicates(subset=["text"])

# Never let a generated sentence be identical to a held-out test sentence.
held_out = {t.lower().strip() for t, _ in hard_tests}
gen_df = gen_df[~gen_df["text"].str.lower().str.strip().isin(held_out)]

print("Generated examples:", len(gen_df))
print(gen_df["label"].value_counts().to_dict())
display(gen_df.sample(8, random_state=SEED))

extra_df = pd.concat([extra_df, gen_df], ignore_index=True)

In [ ]:
# Avoid exact duplicate reviews in the training data.
existing_text = set(
    train_df["text"].astype(str).str.lower().str.strip()
)

extra_df = extra_df[
    ~extra_df["text"].astype(str).str.lower().str.strip().isin(existing_text)
].copy()

# Add the examples to training only.
train_df = pd.concat(
    [train_df, extra_df],
    ignore_index=True
)

x_train = train_df["text"].astype(str).to_numpy()
y_train = train_df["label"].astype("int32").to_numpy()

x_val = val_df["text"].astype(str).to_numpy()
y_val = val_df["label"].astype("int32").to_numpy()

x_test = test_df["text"].astype(str).to_numpy()
y_test = test_df["label"].astype("int32").to_numpy()

print("Final training examples:", len(x_train))
print("Validation examples:", len(x_val))
print("Test examples:", len(x_test))

In [ ]:
MAX_TOKENS = 10000
SEQUENCE_LENGTH = 100

vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    ngrams=(1, 2),
    standardize="lower_and_strip_punctuation"
)

# Learn vocabulary from training data only.
vectorizer.adapt(x_train)

vocabulary = [str(word) for word in vectorizer.get_vocabulary()]

print("Vocabulary size:", len(vocabulary))
print("\nFirst 50 vocabulary entries:")
print(vocabulary[:50])

# Check how the model tokenizes a tricky sentence.
print("\nTokenized example:")
print(
    vectorizer(["I don't think this movie was bad"]).numpy()[0]
)

In [ ]:
text_input = keras.Input(
    shape=(),
    dtype=tf.string,
    name="review_text"
)

# Convert words and phrases to token IDs.
tokens = vectorizer(text_input)

# Learn numerical representations of words.
embedding = keras.layers.Embedding(
    input_dim=len(vocabulary),
    output_dim=64
)(tokens)

# Learn local patterns in the text.
conv = keras.layers.Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu"
)(embedding)

# Retain the strongest detected patterns.
pool = keras.layers.GlobalMaxPooling1D()(conv)

dense = keras.layers.Dense(
    64,
    activation="relu"
)(pool)

dropout = keras.layers.Dropout(0.4)(dense)

# Predict probability of Positive = label 1.
output = keras.layers.Dense(
    1,
    activation="sigmoid"
)(dropout)

model = keras.Model(text_input, output)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stopping]
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Test accuracy:", round(test_accuracy, 4))
print("Test loss:", round(test_loss, 4))

probabilities = model.predict(x_test, verbose=0).ravel()

# 1 = Positive, 0 = Negative.
predictions = (probabilities >= 0.5).astype(int)

print(
    classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Negative", "Positive"],
        zero_division=0
    )
)

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ),
    display_labels=["Negative", "Positive"]
).plot()

plt.show()

In [ ]:
# Evaluate on the held-out sentences defined earlier (never used for training).
test_reviews = np.array([text for text, _ in hard_tests], dtype=object)
test_labels = np.array([label for _, label in hard_tests])

# Safety check: none of them may be in the training data.
train_texts = set(train_df["text"].str.lower().str.strip())
overlap = [t for t in test_reviews if t.lower().strip() in train_texts]
assert not overlap, f"These are in the training data: {overlap}"

scores = model.predict(test_reviews, verbose=0).ravel()
preds = (scores >= 0.5).astype(int)

for text, label, score, pred in zip(test_reviews, test_labels, scores, preds):
    status = "OK  " if pred == label else "MISS"
    print(f"{status} | {score:.3f} | {'Positive' if pred else 'Negative'} | {text}")

print("\nHeld-out accuracy:", round(float((preds == test_labels).mean()), 3),
      f"({int((preds == test_labels).sum())}/{len(test_labels)})")

In [ ]:

custom_sentence = "The movie was not too bad"

prediction = float(model.predict(
    np.array([custom_sentence], dtype=object),
    verbose=0
)[0, 0])

sentiment = "Positive" if prediction >= 0.5 else "Negative"

print("Sentence:", custom_sentence)
print("Prediction:", sentiment)
print("Score:", round(prediction, 4))


## Streamlit app
Run the cells below in order: save the trained model, write `app.py`, then launch it.

In [ ]:
# Save the trained model (includes the text vectorizer).
model.save("sentiment_model.keras")
print("Saved sentiment_model.keras")

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
from pathlib import Path
import keras

st.set_page_config(page_title="Movie Review Sentiment", page_icon="🎬", layout="centered")
MODEL_PATH = Path(__file__).resolve().parent / "sentiment_model.keras"

@st.cache_resource
def load_model(model_path: str):
    return keras.saving.load_model(model_path)

st.title("🎬 Movie Review Sentiment")
st.write("Enter a movie review to estimate whether its sentiment is positive or negative.")
st.caption("This educational model can make mistakes, especially with sarcasm, mixed opinions, and complex negation.")

if not MODEL_PATH.exists():
    st.error("Trained model not found: sentiment_model.keras")
    st.info("Run the notebook from top to bottom to train and save the model, then place the saved file beside app.py.")
    st.stop()

try:
    model = load_model(str(MODEL_PATH))
except Exception as exc:
    st.error("The model could not be loaded. Re-run the notebook with the installed requirements and regenerate sentiment_model.keras.")
    st.exception(exc)
    st.stop()

review = st.text_area("Movie review", placeholder="Example: The acting was excellent, but the story felt slow.", height=140, max_chars=5000)
if st.button("Analyze sentiment", type="primary", use_container_width=True):
    cleaned = review.strip()
    if not cleaned:
        st.warning("Please enter a review first.")
    else:
        with st.spinner("Analyzing review..."):
            score = float(model.predict(np.asarray([cleaned], dtype=object), verbose=0).reshape(-1)[0])
        label = "Positive" if score >= 0.5 else "Negative"
        st.subheader(f"Predicted sentiment: {label}")
        st.metric("Positive sentiment score", f"{score:.1%}")
        st.progress(min(max(score, 0.0), 1.0))
        st.caption("The score is the model's sigmoid output, not a calibrated probability that the prediction is correct.")


In [ ]:
# Start the app from a terminal in this project directory:
# python -m streamlit run app.py
print("App code written to app.py. Start it from a terminal with: python -m streamlit run app.py")

## Reproducibility and limitations

- This notebook expects the original CSV file; it was not included with the uploaded notebook, so it is not bundled here.
- The previously saved notebook output reported 96.67% accuracy on a 90-example random test split and 75% (24/32) on a manually curated challenge set. These are results recorded in the supplied notebook, not a fresh run of this packaged version.
- The custom/generated examples use rule-based wording patterns. This can make the task easier than real-world sentiment and does not establish generalization. Evaluate on a larger, independently sourced dataset before making strong performance claims.
- The challenge sentences are hand-written and useful as a sanity check, but they are too small to be a definitive benchmark.
